In [4]:
from pathlib import Path
import os
import pandas as pd
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
import smtplib

In [5]:
contacts_path = Path('input.csv')
contacts_df = pd.read_csv(contacts_path)
contacts_df

,Name,Email
0,Sabrina's iCloud,sdb5s@icloud.com
1,Sabrina's Scarletmail,sb2071@scarletmail.rutgers.edu


In [9]:
contacts_df['Name'] = contacts_df['Name'].astype(str).str.strip()
contacts_df['Email'] = contacts_df['Email'].astype(str).str.strip()

template_path = Path('test.html')
raw_template = template_path.read_text(encoding='utf-8')

email_template_html = raw_template.replace(
    'Hi <em>Maine News Online</em> editor,',
    'Hi <em>{{name}}</em>,',
)

assert '{{name}}' in email_template_html, 'Name placeholder missing in template.'

image_path = Path('picture.jpeg')
if not image_path.exists():
    raise FileNotFoundError(f'Expected inline image at {image_path!s}')

image_content_id = image_path.name

AssertionError: Name placeholder missing in template.

In [ ]:
def load_smtp_config():
    config = {
        'host': os.environ.get('SMTP_HOST'),
        'port': int(os.environ.get('SMTP_PORT', '587')),
        'username': os.environ.get('SMTP_USERNAME'),
        'password': os.environ.get('SMTP_PASSWORD'),
        'sender': os.environ.get('SMTP_SENDER') or os.environ.get('SMTP_USERNAME'),
        'use_tls': os.environ.get('SMTP_USE_TLS', 'true').lower() in {'1', 'true', 'yes'},
        'use_ssl': os.environ.get('SMTP_USE_SSL', 'false').lower() in {'1', 'true', 'yes'},
    }
    if config['use_ssl'] and config['use_tls']:
        raise ValueError('Configure either TLS or SSL, not both.')
    missing_keys = [key for key in ('host', 'port', 'username', 'password', 'sender') if not config[key]]
    if missing_keys:
        raise ValueError(
            'Missing SMTP configuration values. Set environment variables for: ' + ', '.join(missing_keys)
        )
    return config

SMTP_CONFIG = None
try:
    SMTP_CONFIG = load_smtp_config()
    print('SMTP configuration loaded.')
except ValueError as exc:
    print(exc)

EMAIL_SUBJECT = os.environ.get('EMAIL_SUBJECT', 'THE GLASS EEL - Author Events in Maine')
REPLY_TO = os.environ.get('EMAIL_REPLY_TO')

In [ ]:
def build_email_message(contact, html_template, config):
    name = (contact.get('Name') or '').strip()
    email = (contact.get('Email') or '').strip()
    if not email:
        raise ValueError('Contact is missing an email address.')
    display_name = name or 'there'

    message = MIMEMultipart('related')
    message['Subject'] = EMAIL_SUBJECT
    message['From'] = config['sender']
    message['To'] = email
    if REPLY_TO:
        message['Reply-To'] = REPLY_TO

    alternative = MIMEMultipart('alternative')
    message.attach(alternative)

    plain_body = f'Hi {display_name},
Please view this message in HTML format.'
    alternative.attach(MIMEText(plain_body, 'plain', 'utf-8'))

    html_body = html_template.replace('{{name}}', display_name)
    alternative.attach(MIMEText(html_body, 'html', 'utf-8'))

    with image_path.open('rb') as img_file:
        image_part = MIMEImage(img_file.read())
    image_part.add_header('Content-ID', f'<{image_content_id}>')
    image_part.add_header('Content-Disposition', 'inline', filename=image_path.name)
    message.attach(image_part)

    return message, email, display_name

In [ ]:
if SMTP_CONFIG is None:
    raise RuntimeError('SMTP settings are not configured. Set environment variables and rerun the configuration cell.')

send_results = []
SMTP_CLASS = smtplib.SMTP_SSL if SMTP_CONFIG['use_ssl'] else smtplib.SMTP

with SMTP_CLASS(SMTP_CONFIG['host'], SMTP_CONFIG['port']) as server:
    server.ehlo()
    if SMTP_CONFIG['use_tls'] and not SMTP_CONFIG['use_ssl']:
        server.starttls()
        server.ehlo()
    server.login(SMTP_CONFIG['username'], SMTP_CONFIG['password'])

    for contact in contacts_df.to_dict(orient='records'):
        try:
            message, recipient_email, display_name = build_email_message(contact, email_template_html, SMTP_CONFIG)
            server.sendmail(SMTP_CONFIG['sender'], [recipient_email], message.as_string())
            print(f'[OK] {display_name} <{recipient_email}>')
            send_results.append({'name': display_name, 'email': recipient_email, 'status': 'sent'})
        except Exception as exc:
            name = (contact.get('Name') or '').strip() or (contact.get('Email') or '').strip()
            email = (contact.get('Email') or '').strip()
            print(f'[FAIL] {name} <{email}> :: {exc}')
            send_results.append({'name': name, 'email': email, 'status': f'error: {exc}'})